# 03 — Error Handling

`🟡 Intermediate`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/02_Intermediate/03_error_handling.ipynb)

## 📌 What is it?
Python's exception system lets you catch and handle errors gracefully with `try/except/else/finally`.

## 🧠 Why AI Engineers need this
API calls fail. Rate limits hit. Network timeouts occur. Models return unexpected formats. Robust error handling keeps AI pipelines running reliably.

In [ ]:
# ── TRY / EXCEPT / ELSE / FINALLY ──
import json

def parse_llm_response(raw: str) -> dict:
    """Safely parse a JSON response from an LLM."""
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON parse error: {e}")
        return {}
    except Exception as e:
        print(f"⚠️ Unexpected error: {e}")
        return {}
    else:
        # Runs only if NO exception was raised
        print("✅ Parsed successfully")
        return data
    finally:
        # Always runs — great for cleanup
        print("parse_llm_response() finished")

good = '{"content": "Hello", "tokens": 42}'
bad  = 'not valid json'
print(parse_llm_response(good))
print(parse_llm_response(bad))

In [ ]:
# ── MULTIPLE EXCEPT CLAUSES ──
def safe_api_call(url: str) -> str:
    import urllib.request, socket
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.read().decode()
    except ValueError:
        return "Invalid URL"
    except socket.timeout:
        return "Request timed out"
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

print(safe_api_call("not_a_url"))
print(safe_api_call("https://httpbin.org/delay/10"))

In [ ]:
# ── CUSTOM EXCEPTIONS ──
class APIError(Exception):
    """Base API error."""
    pass

class RateLimitError(APIError):
    """Raised when API rate limit is exceeded."""
    def __init__(self, retry_after=60):
        self.retry_after = retry_after
        super().__init__(f"Rate limit exceeded. Retry after {retry_after}s.")

class AuthenticationError(APIError):
    """Raised when API key is invalid."""
    pass


def call_api(api_key, requests_remaining):
    if not api_key.startswith("sk-"):
        raise AuthenticationError("Invalid API key format")
    if requests_remaining <= 0:
        raise RateLimitError(retry_after=30)
    return {"status": "ok"}

try:
    result = call_api("sk-abc123", requests_remaining=0)
except RateLimitError as e:
    print(f"Rate limited! Wait {e.retry_after}s then retry")
except AuthenticationError:
    print("Check your API key")
except APIError as e:
    print(f"Generic API error: {e}")

In [ ]:
# ── RETRY LOGIC WITH EXPONENTIAL BACKOFF ──
import time, random

def with_retry(func, max_retries=3, base_delay=1.0):
    """Retry a function with exponential backoff."""
    for attempt in range(max_retries):
        try:
            return func()
        except RateLimitError as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            print(f"Attempt {attempt+1} failed. Retrying in {delay:.1f}s...")
            time.sleep(delay)
    
attempt_count = 0
def flaky_call():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise RateLimitError()
    return "success"

result = with_retry(flaky_call)
print(f"Got: {result} after {attempt_count} attempts")

## ⚠️ Common Mistakes
```python
# ❌ Bare except catches EVERYTHING — even KeyboardInterrupt!
try:
    ...
except:
    pass   # silently swallows all errors!

# ✅ Be specific
try:
    ...
except ValueError as e:
    handle(e)
except Exception as e:   # catch remaining, still not bare
    log(e)
    raise             # re-raise after logging!
```

## 🔗 What's Next?
[04 — File I/O →](04_file_io.ipynb)